In [53]:
from VerificationManager import VoiceRecognition, FaceRecognition, RecognitionManager
from os import listdir, path
import pandas as pd
import numpy as np
from tqdm import tqdm

voice_model = VoiceRecognition(lang='en')
face_model = FaceRecognition(model_name='Facenet512')
df = pd.DataFrame(columns=['Name', 'AudioID', 'Face_Centroid', 'Voice_Centroid'])

images_folder = "./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/"

audio_path = "./data/LibriSpeech/train-clean-100/"
audio_files = {}

for user_id in listdir(audio_path):
    if path.isdir(path.join(audio_path, user_id)):
        audio_files[user_id] = []
        for chapter in listdir(path.join(audio_path, user_id)):
            if path.isdir(path.join(audio_path, user_id, chapter)):
                for file in listdir(path.join(audio_path, user_id, chapter)):
                    if file.endswith('.flac'):
                        audio_files[user_id].append(path.join(audio_path, user_id, chapter, file))

Using cache found in /Users/alexd/.cache/torch/hub/snakers4_silero-vad_master


In [54]:
for subject in tqdm(listdir(images_folder)):
    images = listdir(path.join(images_folder, subject))
    if len(images) > 10:
        face_centroid = face_model.enroll_user([path.join(images_folder, subject, img) for img in images[:10]])
        df.loc[len(df)] = {'Name': subject, 'Face_Centroid': face_centroid}

100%|██████████| 5749/5749 [00:16<00:00, 343.34it/s]


In [55]:
# assign to each user a random audio file from the audio_files list
df["Voice_Centroid"] = pd.Series([None] * len(df), dtype=object)

df.AudioID = np.random.choice(list(audio_files.keys()), size=len(df), replace=False)

for i in tqdm(range(len(df))):
    audioID = df.iloc[i]['AudioID']
    voice_centroid = None
    while voice_centroid is None or np.isnan(voice_centroid).any():
        audio_file = np.random.choice(audio_files[audioID])
        try:
            voice_centroid = voice_model.enroll_user([audio_file])
        except:
            voice_centroid = None
    df.at[i, 'Voice_Centroid'] = voice_centroid

100%|██████████| 143/143 [00:46<00:00,  3.06it/s]


In [56]:
#save df to csv
df.to_csv('model_eval_df.csv', index=False)
df = pd.read_csv('model_eval_df.csv')

In [57]:
cnt = 0
for user in audio_files:
     cnt += len(user)
cnt

937

In [58]:
cnt = 0
for user in listdir("./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/"):
    images = listdir(path.join("./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/", user))
    cnt += len(images)
cnt

13233

In [59]:
len(listdir("./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/"))

5749

In [47]:
from os import listdir, path
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Your existing model class (do NOT modify) ---
from VerificationManager import FaceRecognition

# ----------------------------
# Config
# ----------------------------
CSV_PATH = "model_eval_df.csv"  # update if needed
IMAGES_FOLDER = "./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/"

MODEL_NAME = "Facenet512"
TOPK = 5

# Open-set target FAR for threshold selection
TARGET_FAR = 1e-2   # 1% (try 1e-3 if you have lots of unknown probes)
RNG_SEED = 123

# ----------------------------
# Helpers
# ----------------------------
def parse_centroid_str(s: str) -> np.ndarray:
    # CSV centroid strings look like: "[1.23e-3  ... \n  ...]"
    s = s.strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    arr = np.fromstring(s, sep=" ", dtype=np.float32)  # whitespace + newlines ok
    if arr.size == 0:
        raise ValueError("Failed to parse centroid from string.")
    return arr

def build_enrolled_centroids(df: pd.DataFrame):
    """
    FaceRecognition.verify expects centroids as:
      Sequence[Tuple[int, embedding_blob]]
    and it does: np.frombuffer(embedding_blob, dtype=np.float32)
    So we must provide float32 bytes.
    """
    id_to_name = {}
    centroids = []
    for user_id, row in enumerate(df.itertuples(index=False)):
        name = row.Name
        emb = parse_centroid_str(row.Face_Centroid).astype(np.float32)
        centroids.append((user_id, emb.tobytes()))
        id_to_name[user_id] = name
    name_to_id = {v: k for k, v in id_to_name.items()}
    return centroids, id_to_name, name_to_id

def list_images_for_identity(images_folder: str, identity: str):
    d = path.join(images_folder, identity)
    if not path.isdir(d):
        return []
    files = [path.join(d, f) for f in listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    files.sort()
    return files

# ----------------------------
# Load enrolled users
# ----------------------------
df = pd.read_csv(CSV_PATH)
enrolled_names = set(df["Name"].tolist())

centroids, id_to_name, name_to_id = build_enrolled_centroids(df)

# ----------------------------
# Build probe sets
# ----------------------------
known_probes = []    # list of (img_path, true_id)
unknown_probes = []  # list of img_path

all_identities = [d for d in listdir(IMAGES_FOLDER) if path.isdir(path.join(IMAGES_FOLDER, d))]

# Known probes: for enrolled identities, use images beyond the first 5 (since you used [:5] for enrollment)
for name in enrolled_names:
    imgs = list_images_for_identity(IMAGES_FOLDER, name)
    if len(imgs) <= 5:
        continue
    true_id = name_to_id[name]
    for img in imgs[5:]:
        known_probes.append((img, true_id))

# Unknown probes: sample identities NOT enrolled
rng = np.random.default_rng(RNG_SEED)
unknown_identities = [i for i in all_identities if i not in enrolled_names]

# You can control how many unknown identities/images you test:
# Here: sample up to 1000 unknown identities, 1 image each (adjust as you like)
rng.shuffle(unknown_identities)
unknown_identities = unknown_identities[:1000]

for ident in unknown_identities:
    imgs = list_images_for_identity(IMAGES_FOLDER, ident)
    if not imgs:
        continue
    unknown_probes.append(rng.choice(imgs))

print(f"Known probes:   {len(known_probes)}")
print(f"Unknown probes: {len(unknown_probes)}")

# ----------------------------
# Run recognition once (store scores)
# ----------------------------
face_model = FaceRecognition(model_name=MODEL_NAME)

# Store: (is_known, is_correct, top1_score)
known_scores = []
unknown_scores = []
errors = 0

# Known
for img_path, true_id in tqdm(known_probes, desc="Evaluating known probes"):
    try:
        topk = face_model.verify(img_path, centroids=centroids)  # [(pred_id, score), ...]
        pred_id, score = topk[0]
        known_scores.append((pred_id == true_id, float(score)))
    except Exception:
        errors += 1

# Unknown
for img_path in tqdm(unknown_probes, desc="Evaluating unknown probes"):
    try:
        topk = face_model.verify(img_path, centroids=centroids)
        pred_id, score = topk[0]
        unknown_scores.append(float(score))
    except Exception:
        errors += 1

print(f"Errors: {errors}")

# Basic closed-set Rank-1 / Rank-5 (known only)
# Rank-5 requires re-running verify or storing full topk; easiest: compute now quickly by re-running.
rank1 = np.mean([c for (c, _) in known_scores]) if known_scores else float("nan")
print(f"Rank-1 (known, closed-set): {rank1:.2%}")

# ----------------------------
# Threshold sweep for open-set metrics
# ----------------------------
known_correct = np.array([c for (c, s) in known_scores], dtype=bool)
known_top1 = np.array([s for (c, s) in known_scores], dtype=float)
unk_top1 = np.array(unknown_scores, dtype=float)

def metrics_at_threshold(thr: float):
    # Accept if top-1 similarity >= thr
    known_accept = known_top1 >= thr
    unk_accept = unk_top1 >= thr

    FAR = float(np.mean(unk_accept)) if len(unk_top1) else float("nan")
    TAR = float(np.mean(known_accept)) if len(known_top1) else float("nan")  # accept rate for knowns
    DIR_overall = float(np.mean(known_accept & known_correct)) if len(known_top1) else float("nan")
    FRR = 1.0 - TAR if TAR == TAR else float("nan")
    TNR = 1.0 - FAR if FAR == FAR else float("nan")
    return FAR, TNR, TAR, FRR, DIR_overall

# Use observed scores as candidate thresholds (gives exact FAR steps)
grid = np.unique(np.concatenate([known_top1, unk_top1]))
grid = np.concatenate(([0.0], grid, [1.0]))

best_thr = None
best_dir = -1.0

for thr in grid:
    FAR, _, _, _, DIR = metrics_at_threshold(float(thr))
    if FAR <= TARGET_FAR and DIR > best_dir:
        best_dir = DIR
        best_thr = float(thr)

if best_thr is None:
    # Can't achieve that FAR; pick threshold that minimizes FAR (highest thr)
    best_thr = float(np.max(grid))
    best_dir = metrics_at_threshold(best_thr)[-1]

FAR, TNR, TAR, FRR, DIR = metrics_at_threshold(best_thr)

print("\nOpen-set operating point:")
print(f"Target FAR: {TARGET_FAR:g}")
print(f"Chosen threshold: {best_thr:.4f}")
print(f"FAR (unknown accepted): {FAR:.4%} | TNR: {TNR:.4%}")
print(f"TAR (known accepted):   {TAR:.2%} | FRR: {FRR:.2%}")
print(f"DIR overall (known accepted & correct): {DIR:.2%}")


Known probes:   3459
Unknown probes: 1000


Evaluating unknown probes: 100%|██████████| 1000/1000 [01:33<00:00, 10.65it/s]


Errors: 0
Rank-1 (known, closed-set): 92.80%

Open-set operating point:
Target FAR: 0.01
Chosen threshold: 0.8859
FAR (unknown accepted): 1.0000% | TNR: 99.0000%
TAR (known accepted):   62.30% | FRR: 37.70%
DIR overall (known accepted & correct): 61.98%


In [60]:
from os import listdir, path
import numpy as np
import pandas as pd
from tqdm import tqdm

from VerificationManager import VoiceRecognition

# ----------------------------
# Config
# ----------------------------
CSV_PATH = "model_eval_df.csv"
AUDIO_PATH = "./data/LibriSpeech/train-clean-100/"  # same you used
LANG = "en"

# unknown sampling
UNKNOWN_SPEAKERS_TO_SAMPLE = 400   # increase for more stable FAR estimates
UNKNOWN_UTTS_PER_SPEAKER = 3       # increase for more probes

# operating point
TARGET_FAR = 1e-2   # 1% (with few unknown probes, 0.1% is hard to measure reliably)
RNG_SEED = 123

# ----------------------------
# Helpers
# ----------------------------
def parse_centroid_str(s: str) -> np.ndarray:
    s = str(s).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    arr = np.fromstring(s, sep=" ", dtype=np.float32)
    if arr.size == 0:
        raise ValueError("Failed to parse centroid from string.")
    return arr

def list_flacs_for_speaker(audio_root: str, speaker_id: str):
    """Return all .flac files for a LibriSpeech speaker folder."""
    spk_dir = path.join(audio_root, speaker_id)
    if not path.isdir(spk_dir):
        return []
    out = []
    for chapter in listdir(spk_dir):
        chap_dir = path.join(spk_dir, chapter)
        if not path.isdir(chap_dir):
            continue
        for f in listdir(chap_dir):
            if f.endswith(".flac"):
                out.append(path.join(chap_dir, f))
    out.sort()
    return out

def build_centroids_from_csv(df: pd.DataFrame):
    """
    centroids format expected by verify():
      Sequence[Tuple[int, embedding_blob]]
    and VoiceRecognition.verify does: np.frombuffer(embedding_blob, dtype=np.float32)
    """
    id_to_name = {}
    id_to_audioid = {}
    centroids = []

    for user_id, row in enumerate(df.itertuples(index=False)):
        name = row.Name
        audioid = str(int(row.AudioID)) if not isinstance(row.AudioID, str) else row.AudioID
        emb = parse_centroid_str(row.Voice_Centroid).astype(np.float32)

        centroids.append((user_id, emb.tobytes()))
        id_to_name[user_id] = name
        id_to_audioid[user_id] = audioid

    return centroids, id_to_name, id_to_audioid

# ----------------------------
# Load enrolled users/centroids
# ----------------------------
df = pd.read_csv(CSV_PATH)

# Keep only rows with a Voice_Centroid present
df = df[df["Voice_Centroid"].notna()].reset_index(drop=True)

centroids, id_to_name, id_to_audioid = build_centroids_from_csv(df)

enrolled_audioids = set(id_to_audioid.values())
print("Enrolled identities:", len(df))
print("Unique enrolled AudioID:", len(enrolled_audioids))

# ----------------------------
# Build probe sets from LibriSpeech
# ----------------------------
rng = np.random.default_rng(RNG_SEED)

all_speaker_ids = [d for d in listdir(AUDIO_PATH) if path.isdir(path.join(AUDIO_PATH, d))]
unknown_speaker_ids = [s for s in all_speaker_ids if s not in enrolled_audioids]

# Known probes: all utterances of enrolled speakers
# (Note: ideally exclude the specific enrollment utterances; you don't store them currently,
#  so this is "mostly disjoint" but may include the one used for enrollment if it repeats.)
known_probes = []  # list of (flac_path, true_user_id)
for user_id, audioid in id_to_audioid.items():
    flacs = list_flacs_for_speaker(AUDIO_PATH, audioid)
    for f in flacs:
        known_probes.append((f, user_id))

# Unknown probes: sample non-enrolled speakers
rng.shuffle(unknown_speaker_ids)
unknown_speaker_ids = unknown_speaker_ids[:UNKNOWN_SPEAKERS_TO_SAMPLE]

unknown_probes = []
for spk in unknown_speaker_ids:
    flacs = list_flacs_for_speaker(AUDIO_PATH, spk)
    if not flacs:
        continue
    # sample a few utterances per unknown speaker
    take = min(UNKNOWN_UTTS_PER_SPEAKER, len(flacs))
    picks = rng.choice(flacs, size=take, replace=False)
    unknown_probes.extend(list(picks))

print(f"Known probes:   {len(known_probes)}")
print(f"Unknown probes: {len(unknown_probes)}")

# ----------------------------
# Run verification once, store top-1
# ----------------------------
voice_model = VoiceRecognition(lang=LANG)

# store: (has_score, is_correct, top1_score)
known_rows = []
unknown_scores = []
errors = 0

# Known
for flac_path, true_user_id in tqdm(known_probes, desc="Evaluating known probes"):
    try:
        topk = voice_model.verify(flac_path, centroids=centroids)
        if not topk:
            # too short speech -> rejection / no score
            known_rows.append((False, False, np.nan))
            continue
        pred_id, score = topk[0]
        known_rows.append((True, pred_id == true_user_id, float(score)))
    except Exception:
        errors += 1

# Unknown
for flac_path in tqdm(unknown_probes, desc="Evaluating unknown probes"):
    try:
        topk = voice_model.verify(flac_path, centroids=centroids)
        if not topk:
            # treat as rejected (score = -inf)
            unknown_scores.append(-np.inf)
            continue
        pred_id, score = topk[0]
        unknown_scores.append(float(score))
    except Exception:
        errors += 1

print("Errors:", errors)

known_has = np.array([h for (h, c, s) in known_rows], dtype=bool)
known_correct = np.array([c for (h, c, s) in known_rows], dtype=bool)
known_score = np.array([s for (h, c, s) in known_rows], dtype=float)
unk_score = np.array(unknown_scores, dtype=float)

# Rank-1 (closed-set): only where we have a score (speech>=2s)
rank1 = float(np.mean(known_correct[known_has])) if np.any(known_has) else float("nan")
coverage = float(np.mean(known_has)) if len(known_has) else float("nan")
print(f"Rank-1 (known, among scorable): {rank1:.2%}")
print(f"Coverage (known with >=2s speech): {coverage:.2%}")

# ----------------------------
# Open-set metrics (threshold on top-1 score)
# Accept if has_score and score >= thr
# ----------------------------
def metrics_at_threshold(thr: float):
    known_accept = known_has & (known_score >= thr)
    unk_accept = (unk_score >= thr)  # unk_score is -inf when too short -> rejected

    FAR = float(np.mean(unk_accept)) if len(unk_score) else float("nan")
    TNR = 1.0 - FAR if FAR == FAR else float("nan")

    TAR = float(np.mean(known_accept)) if len(known_accept) else float("nan")
    FRR = 1.0 - TAR if TAR == TAR else float("nan")

    DIR_overall = float(np.mean(known_accept & known_correct)) if len(known_accept) else float("nan")
    return FAR, TNR, TAR, FRR, DIR_overall

grid = np.unique(np.concatenate([known_score[np.isfinite(known_score)], unk_score[np.isfinite(unk_score)]]))
grid = np.concatenate(([-np.inf], grid, [1.0]))

best_thr = None
best_dir = -1.0
for thr in grid:
    FAR, _, _, _, DIR = metrics_at_threshold(float(thr))
    if FAR <= TARGET_FAR and DIR > best_dir:
        best_dir = DIR
        best_thr = float(thr)

if best_thr is None:
    best_thr = float(np.max(grid))
    best_dir = metrics_at_threshold(best_thr)[-1]

FAR, TNR, TAR, FRR, DIR = metrics_at_threshold(best_thr)
print("\nOpen-set operating point:")
print(f"Target FAR: {TARGET_FAR:g}")
print(f"Chosen threshold: {best_thr:.4f}")
print(f"FAR (unknown accepted): {FAR:.4%} | TNR: {TNR:.4%}")
print(f"TAR (known accepted):   {TAR:.2%} | FRR: {FRR:.2%}")
print(f"DIR overall (known accepted & correct): {DIR:.2%}")


Enrolled identities: 143
Unique enrolled AudioID: 143
Known probes:   16277
Unknown probes: 324


Using cache found in /Users/alexd/.cache/torch/hub/snakers4_silero-vad_master
Evaluating unknown probes: 100%|██████████| 324/324 [02:11<00:00,  2.46it/s]


Errors: 0
Rank-1 (known, among scorable): 99.49%
Coverage (known with >=2s speech): 99.14%

Open-set operating point:
Target FAR: 0.01
Chosen threshold: 0.9098
FAR (unknown accepted): 0.9259% | TNR: 99.0741%
TAR (known accepted):   62.33% | FRR: 37.67%
DIR overall (known accepted & correct): 62.33%


In [65]:
%load_ext autoreload
%autoreload 2

import os
from os import path, listdir
import numpy as np
import pandas as pd
from tqdm import tqdm

from VerificationManager import FaceRecognition, VoiceRecognition, RecognitionManager

# ----------------------------
# Paths
# ----------------------------
CSV_PATH = "model_eval_df.csv"
IMAGES_FOLDER = "./data/LFW Dataset/lfw-deepfunneled/lfw-deepfunneled/"
AUDIO_PATH = "./data/LibriSpeech/train-clean-100/"

DB_PATH = "combined_eval.db"  # new db per run

RNG_SEED = 123
rng = np.random.default_rng(RNG_SEED)

# ----------------------------
# Helpers: list files
# ----------------------------
def list_lfw_images(person_name: str):
    d = path.join(IMAGES_FOLDER, person_name)
    if not path.isdir(d): return []
    imgs = [path.join(d, f) for f in listdir(d) if f.lower().endswith((".jpg",".jpeg",".png"))]
    imgs.sort()
    return imgs

def list_librispeech_flacs(speaker_id: str):
    d = path.join(AUDIO_PATH, str(speaker_id))
    if not path.isdir(d): return []
    out = []
    for chapter in listdir(d):
        cd = path.join(d, chapter)
        if not path.isdir(cd): continue
        for f in listdir(cd):
            if f.endswith(".flac"):
                out.append(path.join(cd, f))
    out.sort()
    return out

# ----------------------------
# Load enrolled mapping from CSV
# ----------------------------
df = pd.read_csv(CSV_PATH)
df = df[df["Voice_Centroid"].notna() & df["Face_Centroid"].notna()].reset_index(drop=True)

enrolled_names = df["Name"].tolist()
enrolled_audioids = df["AudioID"].astype(str).tolist()

# ----------------------------
# Build RecognitionManager (fresh DB)
# ----------------------------
if path.exists(DB_PATH):
    os.remove(DB_PATH)

methods = {
    "face_verification": FaceRecognition(model_name="Facenet512"),
    "voice_verification": VoiceRecognition(lang="en")
}
rm = RecognitionManager(methods=methods, enrolled_users_db_name=DB_PATH)

# Enroll users into DB using centroids already in CSV?
# IMPORTANT: RecognitionManager.enroll_user expects raw examples, not centroids.
# So for combined eval, you should enroll using actual files (like you did in your data build),
# OR reuse the same enrollment process that produced the CSV.
#
# Here is the "re-enroll from files" approach consistent with your earlier experiments:
ENROLL_FACE_IMAGES = 10  # or 15, whatever you chose
for name, audioid in tqdm(list(zip(enrolled_names, enrolled_audioids)), desc="Enrolling users in DB"):
    imgs = list_lfw_images(name)
    if len(imgs) < ENROLL_FACE_IMAGES:
        continue
    # enroll face from first N images
    face_examples = imgs[:ENROLL_FACE_IMAGES]

    # enroll voice from 1 random file (you can do more if you want)
    flacs = list_librispeech_flacs(audioid)
    if not flacs:
        continue
    voice_example = rng.choice(flacs)

    rm.enroll_user(name, {
        "face_verification": face_examples,
        "voice_verification": [voice_example]
    })

# Map DB ids to names (for ground truth)
id_name_pairs = rm.get_user_list()
name_to_dbid = {name: uid for (uid, name) in id_name_pairs}

# ----------------------------
# Build probe sets (KNOWN)
# ----------------------------
known_trials = []
for name, audioid in zip(enrolled_names, enrolled_audioids):
    if name not in name_to_dbid:
        continue
    uid = name_to_dbid[name]

    imgs = list_lfw_images(name)
    flacs = list_librispeech_flacs(audioid)

    # Known face probes: use images beyond enrollment range
    face_probes = imgs[ENROLL_FACE_IMAGES:] if len(imgs) > ENROLL_FACE_IMAGES else []
    if not face_probes or not flacs:
        continue

    # Sample a manageable number to control runtime
    for img in rng.choice(face_probes, size=min(5, len(face_probes)), replace=False):
        # sample a few audios too
        for aud in rng.choice(flacs, size=min(3, len(flacs)), replace=False):
            known_trials.append((uid, name, img, aud))

print("Known trials:", len(known_trials))

# ----------------------------
# Build UNKNOWN trials (3 scenarios)
# ----------------------------
# Non-enrolled pools
all_lfw_ids = [d for d in listdir(IMAGES_FOLDER) if path.isdir(path.join(IMAGES_FOLDER, d))]
non_enrolled_lfw = [x for x in all_lfw_ids if x not in set(enrolled_names)]

all_spk_ids = [d for d in listdir(AUDIO_PATH) if path.isdir(path.join(AUDIO_PATH, d))]
non_enrolled_spk = [x for x in all_spk_ids if x not in set(map(str, enrolled_audioids))]

def sample_unknown_face():
    ident = rng.choice(non_enrolled_lfw)
    imgs = list_lfw_images(ident)
    return rng.choice(imgs) if imgs else None

def sample_unknown_voice():
    spk = rng.choice(non_enrolled_spk)
    flacs = list_librispeech_flacs(spk)
    return rng.choice(flacs) if flacs else None

unknown_trials = {
    "unknown_both": [],
    "known_face_unknown_voice": [],
    "known_voice_unknown_face": []
}

# sample counts (keep small; voice eval is slow)
N_UNKNOWN = 300

# unknown_both
for _ in range(N_UNKNOWN):
    img = sample_unknown_face()
    aud = sample_unknown_voice()
    if img and aud:
        unknown_trials["unknown_both"].append((img, aud))

# mixed scenarios
known_subset = rng.choice(known_trials, size=min(200, len(known_trials)), replace=False)
for (uid, name, img, aud) in known_subset:
    u_aud = sample_unknown_voice()
    u_img = sample_unknown_face()
    if u_aud:
        unknown_trials["known_face_unknown_voice"].append((img, u_aud))
    if u_img:
        unknown_trials["known_voice_unknown_face"].append((u_img, aud))

for k in unknown_trials:
    print(k, len(unknown_trials[k]))

# ----------------------------
# Run trials
# ----------------------------
from collections import Counter, defaultdict

def run_one(img_path, aud_path):
    return rm.verify({
        "face_verification": img_path,
        "voice_verification": aud_path
    })

# Known evaluation
known_accept = 0
known_correct = 0
known_reason = Counter()

for uid, name, img, aud in tqdm(known_trials, desc="Combined: known"):
    out = run_one(img, aud)
    if out.get("result") == "accept":
        known_accept += 1
        # predicted id from voice_match (gated)
        pred_id = out["voice_match"][0]
        if pred_id == uid:
            known_correct += 1
    else:
        known_reason[out.get("reason", "reject_unknown_reason")] += 1

known_total = len(known_trials)
TAR = known_accept / known_total if known_total else float("nan")
DIR = known_correct / known_total if known_total else float("nan")

print("\nKNOWN:")
print("Trials:", known_total)
print(f"TAR (accepted): {TAR:.2%}")
print(f"DIR overall (accepted & correct): {DIR:.2%}")
print("Reject reasons:", known_reason)

# Unknown evaluation (per scenario)
def eval_unknown(trials):
    acc = 0
    reasons = Counter()
    for img, aud in tqdm(trials, desc="Combined: unknown"):
        out = run_one(img, aud)
        if out.get("result") == "accept":
            acc += 1
        else:
            reasons[out.get("reason", "reject_unknown_reason")] += 1
    total = len(trials)
    FAR = acc / total if total else float("nan")
    return FAR, reasons

print("\nUNKNOWN:")
for scenario, trials in unknown_trials.items():
    FAR, reasons = eval_unknown(trials)
    print(f"{scenario}: FAR={FAR:.2%} | rejects={dict(reasons)}")


Using cache found in /Users/alexd/.cache/torch/hub/snakers4_silero-vad_master


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Enrolling users in DB: 100%|██████████| 143/143 [00:57<00:00,  2.48it/s]


Known trials: 1767
unknown_both 300
known_face_unknown_voice 200
known_voice_unknown_face 200


Combined: known: 100%|██████████| 1767/1767 [13:47<00:00,  2.13it/s]



KNOWN:
Trials: 1767
TAR (accepted): 96.21%
DIR overall (accepted & correct): 96.21%
Reject reasons: Counter({'best voice match not in face verification results': 49, 'not enough speech detected': 14, 'voice verification below threshold': 4})

UNKNOWN:


... (2 outputs left out)

Combined: unknown: 100%|██████████| 200/200 [01:31<00:00,  2.18it/s]


known_face_unknown_voice: FAR=2.00% | rejects={'voice verification below threshold': 67, 'best voice match not in face verification results': 128, 'not enough speech detected': 1}


Combined: unknown: 100%|██████████| 200/200 [01:31<00:00,  2.19it/s]

known_voice_unknown_face: FAR=2.50% | rejects={'best voice match not in face verification results': 190, 'not enough speech detected': 3, 'voice verification below threshold': 2}
